# Prompt 1 - Direct Schema-Aware Prompting

This notebook contains the implementation of the first prompting strategy used in the homework. It evaluates Qwen on the reduced `dataset_30` benchmark using a direct schema-aware prompt, where the model receives the full graph schema and generates the Cypher query in a single step.


In [ ]:
!pip -q install -U transformers accelerate sentencepiece gdown


In [ ]:
import os
import gdown

os.makedirs("/content/data", exist_ok=True)

LINK_DATASET_30 = "https://drive.google.com/uc?id=1IOvn9rx5-hFES6PU-3Ow8th5h-cl4z6v"
DATASET_PATH = "/content/data/dataset_30.csv"

gdown.download(LINK_DATASET_30, DATASET_PATH, quiet=False)
print("Dataset saved to:", DATASET_PATH)


In [ ]:
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import torch
from transformers import pipeline

DATASET_PATH = "/content/data/dataset_30.csv"
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
SLEEP_SECONDS = 0.1

FULL_SCHEMA = 'Node properties:\nMovie {posterEmbedding: LIST, url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, plotEmbedding: LIST, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}\nGenre {name: STRING}\nUser {userId: STRING, name: STRING}\nActor {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nDirector {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nPerson {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nRelationship properties:\nRATED {rating: FLOAT, timestamp: INTEGER}\nACTED_IN {role: STRING}\nDIRECTED {role: STRING}\nThe relationships:\n(:Movie)-[:IN_GENRE]->(:Genre)\n(:User)-[:RATED]->(:Movie)\n(:Actor)-[:ACTED_IN]->(:Movie)\n(:Actor)-[:DIRECTED]->(:Movie)\n(:Director)-[:DIRECTED]->(:Movie)\n(:Director)-[:ACTED_IN]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)\n(:Person)-[:DIRECTED]->(:Movie)'

SYSTEM_PROMPT = (
    "You are an expert Neo4j and Cypher assistant. "
    "Always answer with a single valid JSON object and nothing else. "
    "Never use SQL syntax such as GROUP BY, HAVING, JOIN, or SELECT. "
    "Use Cypher WITH for aggregation steps. "
    "Use only schema-valid labels, relationship types, and properties."
)

PIPE = None

def load_generation_pipeline(model_name: str = MODEL_NAME):
    global PIPE
    if PIPE is not None:
        return PIPE

    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    PIPE = pipeline(
        "text-generation",
        model=model_name,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    if PIPE.tokenizer.pad_token_id is None:
        PIPE.tokenizer.pad_token_id = PIPE.tokenizer.eos_token_id
    return PIPE

def call_model(prompt: str, max_new_tokens: int) -> str:
    pipe = load_generation_pipeline()
    outputs = pipe(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=pipe.tokenizer.pad_token_id,
    )
    generated = outputs[0]["generated_text"]
    text = generated[-1]["content"].strip() if isinstance(generated, list) else str(generated).strip()
    time.sleep(SLEEP_SECONDS)
    return text

def extract_json_block(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None

    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        parsed = json.loads(cleaned)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    if start == -1:
        return None

    depth = 0
    for index in range(start, len(cleaned)):
        char = cleaned[index]
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                candidate = cleaned[start : index + 1]
                try:
                    parsed = json.loads(candidate)
                    return parsed if isinstance(parsed, dict) else None
                except json.JSONDecodeError:
                    return None
    return None

def normalize_cypher(text: Any) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    return text

def read_dataset(dataset_path: str = DATASET_PATH) -> pd.DataFrame:
    df = pd.read_csv(dataset_path)
    required = {"id", "question", "gold_cypher"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
    return df

def get_completed_ids(output_path: str) -> set[str]:
    path = Path(output_path)
    if not path.exists():
        return set()
    df = pd.read_csv(path)
    if "id" not in df.columns:
        return set()
    return set(df["id"].astype(str))

def append_result(output_path: str, result: Dict[str, Any]) -> None:
    row_df = pd.DataFrame([result])
    header = not Path(output_path).exists()
    row_df.to_csv(output_path, mode="a", header=header, index=False)


In [ ]:
OUTPUT_PATH = "/content/results_prompt1_qwen_dataset30.csv"
MAX_NEW_TOKENS = 220

FEW_SHOTS = '''
Example 1
Question: Which movies did user Alice rate?
Cypher:
MATCH (u:User {name: "Alice"})-[:RATED]->(m:Movie)
RETURN m.title

Example 2
Question: What are the top 3 longest movies by runtime?
Cypher:
MATCH (m:Movie)
RETURN m.title, m.runtime
ORDER BY m.runtime DESC
LIMIT 3

Example 3
Question: Which actors starred in movies with a budget above 200 million?
Cypher:
MATCH (a:Actor)-[:ACTED_IN]->(m:Movie)
WHERE m.budget > 200000000
RETURN a.name, m.title, m.budget

Example 4
Question: What genres does the movie "Toy Story" belong to?
Cypher:
MATCH (m:Movie {title: "Toy Story"})-[:IN_GENRE]->(g:Genre)
RETURN g.name

Example 5
Question: Which 5 movies have been rated by the highest number of users?
Cypher:
MATCH (u:User)-[:RATED]->(m:Movie)
WITH m, COUNT(u) AS numUsers
ORDER BY numUsers DESC
LIMIT 5
RETURN m.title AS MovieTitle, numUsers
'''.strip()

def build_prompt(question: str) -> str:
    return f"""
Task:
Translate the natural language question into a correct Cypher query for the given graph schema.

Instructions:
- Use only labels, relationships, and properties present in the schema.
- Do not invent schema elements.
- Do not output markdown fences.
- Do not use SQL keywords such as GROUP BY, HAVING, JOIN, or SELECT.
- Return a compact but correct query that answers the question.

Schema:
{FULL_SCHEMA}

Few-shot examples:
{FEW_SHOTS}

Question:
{question}

Return JSON:
{{
  "reasoning": "short explanation",
  "cypher": "final Cypher query"
}}
""".strip()

def run_prompt1(resume: bool = True) -> pd.DataFrame:
    df = read_dataset()
    completed_ids = get_completed_ids(OUTPUT_PATH) if resume else set()

    for _, row in df.iterrows():
        if str(row["id"]) in completed_ids:
            continue

        raw_output = ""
        parse_ok = False
        reasoning = ""
        predicted_cypher = ""
        error_message = ""

        try:
            raw_output = call_model(build_prompt(row["question"]), max_new_tokens=MAX_NEW_TOKENS)
            parsed = extract_json_block(raw_output)
            if parsed is None:
                error_message = "Could not parse model output as JSON."
            else:
                reasoning = str(parsed.get("reasoning", "")).strip()
                predicted_cypher = str(parsed.get("cypher", "")).strip()
                parse_ok = bool(predicted_cypher)
                if not parse_ok:
                    error_message = "JSON parsed but cypher field is empty."
        except Exception as exc:
            error_message = str(exc)

        result = {
            "id": row["id"],
            "difficulty": row.get("difficulty", ""),
            "source_type": row.get("source_type", ""),
            "source_row": row.get("source_row", ""),
            "question": row["question"],
            "gold_cypher": row.get("gold_cypher", ""),
            "prompt_name": "prompt_1_mind_the_query_qwen",
            "model_name": MODEL_NAME,
            "parse_ok": parse_ok,
            "exact_match": normalize_cypher(predicted_cypher) == normalize_cypher(row["gold_cypher"]),
            "reasoning": reasoning,
            "predicted_cypher": predicted_cypher,
            "raw_output": raw_output,
            "error_message": error_message,
        }
        append_result(OUTPUT_PATH, result)
        print(f"[{row['id']}] parse_ok={parse_ok} exact_match={result['exact_match']}")

    return pd.read_csv(OUTPUT_PATH)

df_results = run_prompt1(resume=True)
df_results.head()
